# 2. LLM Bootstrap Labels

Runs `ai_classify` for every taxonomy whose strategy is `'ai_classify'`
(small/medium taxonomies, ≤500 leaves: `three_level`, `gl_map`).  Writes
results into the unified `cat_predictions` table.

UNSPSC is far too large for `ai_classify` and is handled separately by
vector search in notebook 3.

Why `ai_classify`?
- Single SQL primitive, no prompt engineering required.
- Returns a label drawn from a closed set, so it can't hallucinate codes.
- Cheap and parallelizes across the warehouse.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openai openpyxl pydantic
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import STRATEGY, list_schemas

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('invoices_table', config.invoices)
dbutils.widgets.text('limit_rows', '0')   # 0 = all

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
invoices_table = dbutils.widgets.get('invoices_table')
limit_rows = int(dbutils.widgets.get('limit_rows'))
spark.sql(f'USE {catalog}.{schema}')

In [ ]:
"""ai_query bootstrap on three_level: returns label + confidence in a single call.

We previously used hierarchical ``ai_classify`` (L1 then L2) which produced
labels but no confidence. Swapping to ``ai_query`` lets us also collect a
self-reported confidence on the same call. We parse the model's JSON
output with ``from_json`` (some warehouse versions don't honor the
``returnType =>`` shortcut and silently return STRING).

Output is normalized to 0-1 (model returns 1-5; we map (x-1)/4) so the
confidence column is comparable across schemas (catboost predict_proba,
tsvector ts_rank, agent_review).
"""

THREE_LEVEL = "three_level"


def _l2_paths() -> list[tuple[str, str]]:
    pdf = (
        spark.table(f"{catalog}.{schema}.{THREE_LEVEL}_taxonomy")
             .select("level_path").toPandas()
    )
    out: set[tuple[str, str]] = set()
    for _, r in pdf.iterrows():
        lp = r["level_path"]
        items = list(lp) if lp is not None and len(lp) else []
        if len(items) >= 2:
            out.add((items[0], items[1]))
    return sorted(out)


def classify_three_level() -> None:
    paths = _l2_paths()
    options_text = "\n".join(f"- {l1} > {l2}" for l1, l2 in paths)
    limit_clause = f"LIMIT {limit_rows}" if limit_rows > 0 else ""

    prompt_template = (
        "You are categorizing a procurement invoice. "
        "Pick exactly ONE (Level 1, Level 2) pair from the list below "
        "and report a confidence integer 1-5 (1=can't tell, 5=exact match). "
        "Reply ONLY as JSON on a single line:\n"
        '  {"l1":"<exact L1>","l2":"<exact L2>","confidence":<1-5>}\n'
        "No prose, no markdown fences.\n\n"
        f"OPTIONS:\n{options_text}\n\n"
        "Invoice:\n"
    )
    # Escape single quotes for SQL literal embedding.
    prompt_sql = prompt_template.replace("'", "''")

    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW _three_level_raw AS
    SELECT
        i.order_id,
        from_json(
            ai_query(
                '{config.large_llm_endpoint}',
                CONCAT(
                    '{prompt_sql}',
                    'Supplier: ', COALESCE(i.supplier, ''), '\\n',
                    'Description: ', COALESCE(i.description, ''), '\\n',
                    'Cost Centre: ', COALESCE(i.cost_centre, '')
                )
            ),
            'STRUCT<l1: STRING, l2: STRING, confidence: INT>'
        ) AS pred
    FROM {catalog}.{schema}.invoices i
    {limit_clause}
    """)

    # Pick the L3 deterministically: first L3 under the predicted L2.
    spark.sql(f"""
    MERGE INTO {catalog}.{schema}.cat_predictions t
    USING (
      WITH p AS (
        SELECT order_id, pred.l1 AS pred_l1, pred.l2 AS pred_l2,
               pred.confidence AS conf_raw
        FROM _three_level_raw
        WHERE pred IS NOT NULL AND pred.l1 IS NOT NULL AND pred.l2 IS NOT NULL
      ),
      tax AS (
        SELECT level_path[0] AS l1, level_path[1] AS l2,
               level_path[2] AS l3, code
        FROM {catalog}.{schema}.three_level_taxonomy
      ),
      ranked AS (
        SELECT p.order_id, p.pred_l1, p.pred_l2, p.conf_raw,
               COALESCE(tax.l3, p.pred_l2) AS code,
               ARRAY(p.pred_l1, p.pred_l2, COALESCE(tax.l3, p.pred_l2)) AS lp,
               ROW_NUMBER() OVER (PARTITION BY p.order_id ORDER BY tax.code) AS r
        FROM p LEFT JOIN tax ON tax.l1 = p.pred_l1 AND tax.l2 = p.pred_l2
      )
      SELECT order_id,
             '{THREE_LEVEL}' AS schema_name,
             code,
             code AS label,
             lp AS level_path,
             CASE
               WHEN conf_raw IS NULL THEN NULL
               WHEN conf_raw < 1 THEN 0.0
               WHEN conf_raw > 5 THEN 1.0
               ELSE (CAST(conf_raw AS DOUBLE) - 1.0) / 4.0
             END AS confidence,
             'ai_query (l1+l2 in one call, model self-reported confidence)' AS rationale,
             'ai_classify' AS source,
             ARRAY() AS candidates,
             current_timestamp() AS classified_at
      FROM ranked WHERE r = 1
    ) s
    ON t.order_id = s.order_id AND t.schema_name = s.schema_name AND t.source = s.source
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print("Merged ai_query (label + self-reported confidence) predictions for three_level")


print(f"Bootstrap classifying three_level (ai_query with from_json parsing)…")
classify_three_level()
print("\nNote: gl_map and unspsc are not bootstrapped here.")
print("  - gl_map (292 codes) requires hierarchical batching too -- consider notebook 3 or build incrementally")
print("  - unspsc (~150k codes) uses Lakebase pgvector hybrid search in notebook 3")


In [ ]:
"""Classify gl_map (~292 codes) using parallel chat completions.

Uses src.llm.chat_with_retry which handles 429 / 5xx / connection errors
with full-jitter exponential backoff and honors any Retry-After hints
from the FM endpoint.

The model is asked for a self-reported confidence (1-5) in addition to
the code. We normalize that to a 0-1 confidence so downstream UIs and
the agent reviewer have a usable signal.
"""

import concurrent.futures
import threading
import json
import re
from datetime import datetime
from databricks.sdk import WorkspaceClient

from src.llm import chat_with_retry

GL_SCHEMA = "gl_map"
MAX_WORKERS = 4  # workspace LLM endpoint rate-limits at higher concurrency

_JSON_RE = re.compile(r"\{[^{}]*\}")


def _build_candidates(schema_name: str):
    pdf = (
        spark.table(f"{catalog}.{schema}.{schema_name}_taxonomy")
             .select("code", "level_path", "label").toPandas()
    )
    lines = []
    by_code = {}
    for _, r in pdf.iterrows():
        lp = r["level_path"]
        items = list(lp) if lp is not None and len(lp) else []
        path = " > ".join([str(p) for p in items if p])
        lines.append(f"- {r['code']}: {path or r['label']}")
        by_code[str(r['code'])] = {
            "code": str(r['code']),
            "label": r['label'],
            "level_path": items,
        }
    return "\n".join(lines), by_code


def _extract_code_and_conf(text: str, by_code: dict) -> tuple[str | None, float | None]:
    """Pull a (code, confidence_0_1) tuple from the model output.

    The model is asked for ``{"code": "...", "confidence": 1-5}``. We
    accept several common malformations (markdown fences, trailing
    text, bare token) and fall back to ``confidence=None`` when the
    field is missing rather than failing the whole row.
    """
    if not text:
        return None, None

    def _coerce_conf(v) -> float | None:
        if v is None:
            return None
        try:
            x = float(v)
        except (TypeError, ValueError):
            return None
        # Map 1-5 self-reported scale to 0-1.
        if 0 <= x <= 1:
            return x
        if 1 <= x <= 5:
            return round((x - 1) / 4, 3)
        return None

    for candidate in (text, *(m.group(0) for m in _JSON_RE.finditer(text))):
        try:
            data = json.loads(candidate)
        except Exception:
            continue
        code = str(data.get("code", "")).strip()
        if code in by_code:
            return code, _coerce_conf(data.get("confidence"))

    for tok in re.split(r"[\s,'\"`]+", text):
        if tok in by_code:
            return tok, None
    return None, None


def classify_chat(schema_name: str) -> None:
    cands_text, by_code = _build_candidates(schema_name)
    invoices = spark.sql(f"""
        SELECT order_id,
               CONCAT_WS(' | ', supplier, description, cost_centre) AS text
        FROM {catalog}.{schema}.invoices
        ORDER BY order_id
    """).toPandas()
    print(f"   {len(invoices):,} invoices to classify against {schema_name}")

    client = WorkspaceClient().serving_endpoints.get_open_ai_client()
    system_prompt = (
        "Pick exactly ONE category code from the list below for the invoice. "
        "Reply ONLY with a JSON object on a single line:\n"
        '  {"code": "<exact_code>", "confidence": <integer 1-5>}\n'
        "Confidence scale:\n"
        "  1 = cannot tell, 2 = several plausible, 3 = likely correct,\n"
        "  4 = confident, 5 = exact match.\n"
        "No prose, no explanation.\n\n"
        f"CATEGORIES:\n{cands_text}"
    )

    fail_count = [0]
    retry_count = [0]
    lock = threading.Lock()

    def _on_retry(attempt: int, err: Exception, delay: float) -> None:
        with lock:
            retry_count[0] += 1

    def _classify(text: str):
        try:
            resp = chat_with_retry(
                client,
                model=config.large_llm_endpoint,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": text},
                ],
                temperature=0.0,
                max_tokens=64,
                max_attempts=8,
                base_delay=2.0,
                max_delay=60.0,
                on_retry=_on_retry,
            )
            content = resp.choices[0].message.content or ""
            code, conf = _extract_code_and_conf(content, by_code)
            if code is None:
                with lock:
                    fail_count[0] += 1
                    if fail_count[0] <= 3:
                        print(f"   miss: invoice='{text[:80]}…' got='{content[:80]}'")
            return code, conf
        except Exception as e:
            with lock:
                fail_count[0] += 1
                if fail_count[0] <= 3:
                    print(f"   error: {type(e).__name__}: {str(e)[:120]}")
            return None, None

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        results = list(ex.map(_classify, invoices["text"].tolist()))

    print(f"   misses/errors: {fail_count[0]}, total retries: {retry_count[0]}")

    now = datetime.utcnow()
    rows = []
    for order_id, (code, conf) in zip(invoices["order_id"].tolist(), results):
        if not code:
            continue
        c = by_code[code]
        rows.append({
            "order_id": str(order_id),
            "schema_name": schema_name,
            "code": c["code"],
            "label": c["label"],
            "level_path": c["level_path"],
            "confidence": conf,
            "rationale": "ai chat with full candidate list (self-reported confidence)",
            "source": "ai_classify_chat",
            "candidates": [],
            "classified_at": now,
        })
    print(f"   {len(rows):,} predictions returned")
    if not rows:
        return

    from pyspark.sql.types import (
        StructType, StructField, StringType, ArrayType, DoubleType, TimestampType,
    )
    pred_schema = StructType([
        StructField("order_id", StringType()), StructField("schema_name", StringType()),
        StructField("code", StringType()), StructField("label", StringType()),
        StructField("level_path", ArrayType(StringType())),
        StructField("confidence", DoubleType()), StructField("rationale", StringType()),
        StructField("source", StringType()),
        StructField("candidates", ArrayType(StructType([
            StructField("code", StringType()), StructField("label", StringType()),
            StructField("score", DoubleType()),
        ]))),
        StructField("classified_at", TimestampType()),
    ])
    sdf = spark.createDataFrame(rows, schema=pred_schema)
    sdf.createOrReplaceTempView(f"_chat_{schema_name}")
    spark.sql(f"""
    MERGE INTO {catalog}.{schema}.cat_predictions t
    USING _chat_{schema_name} s
    ON t.order_id = s.order_id AND t.schema_name = s.schema_name AND t.source = s.source
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"   merged ai_classify_chat predictions for {schema_name}")


print("==> Classifying gl_map with parallel chat completions (with retry/backoff)")
classify_chat(GL_SCHEMA)


## Sanity check + 3-level accuracy

In [ ]:
spark.sql(f"""
SELECT schema_name, source, COUNT(*) AS n
FROM {catalog}.{schema}.cat_predictions
GROUP BY schema_name, source ORDER BY schema_name, source
""").display()

spark.sql(f"""
WITH preds AS (
  SELECT order_id, level_path[0] AS pred_l1, level_path[1] AS pred_l2
  FROM {catalog}.{schema}.cat_predictions
  WHERE schema_name = 'three_level' AND source = 'ai_classify'
)
SELECT COUNT(*) AS n,
  ROUND(AVG(CASE WHEN p.pred_l1 = i.category_level_1 THEN 1.0 ELSE 0.0 END), 3) AS l1_acc,
  ROUND(AVG(CASE WHEN p.pred_l2 = i.category_level_2 THEN 1.0 ELSE 0.0 END), 3) AS l2_acc
FROM preds p JOIN {catalog}.{schema}.{invoices_table} i ON p.order_id = i.order_id
""").display()